# STIR-Net V1 — 11 Stage-B, existence, and true-correction diagnostics

This notebook loads the **Notebook-10 step-25 checkpoint** and performs diagnostics only. It does **not** retrain the model or modify source code.

Notebook 10 + Napari established that:
- corrected primary/split matching works;
- many matched masks are non-empty;
- the normal final `0.50` existence filter suppresses nearly everything;
- some primary masks can be genuinely good;
- temporal/discovery render support often misses most of the matched GT;
- coarse masks remain weak;
- dense spatial geometry still collapses during full end-to-end training.

Notebook 11 answers four narrower questions:

1. **Existence bottleneck:** how good is the current mask branch when existence filtering is relaxed or ignored?
2. **True correction:** are primary/split masks actually improving the current CC input, especially merged sources?
3. **Stage-B plausibility:** are temporal/discovery queries being forced onto implausibly distant GT cells?
4. **Existence vs count:** is count consistency materially fighting positive existence supervision, or is the query representation itself poorly separable?

The goal is to decide the next source patch before implementing the staged curriculum.


In [ ]:
from pathlib import Path
import gc, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config, _repo_root, build_real_batch,
)
from learned.stirnet.debugging.probes.matching import run_matching_probe
from learned.stirnet.model.losses import binary_focal_loss_with_logits
from learned.stirnet.model.matcher import (
    build_local_support_masks, target_masks_at_shape, valid_target_indices,
)
from learned.stirnet.model.query_builder import (
    QUERY_PRIMARY, QUERY_SPLIT, QUERY_TEMPORAL, QUERY_DISCOVERY,
)
from learned.stirnet.training.checkpoint import load_checkpoint
from learned.stirnet.training.trainer import move_to_device

SEED = 40266
AMP_DTYPE = torch.float16

QUERY_NAMES = {
    QUERY_PRIMARY: "primary",
    QUERY_SPLIT: "split",
    QUERY_TEMPORAL: "temporal",
    QUERY_DISCOVERY: "discovery",
}

REPO_ROOT = _repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data" / "learned" / "stirnet" / "first_overfit" / "BlastoSPIM1_F22_030_034"
RUN10_DIR = REPO_ROOT / "runs" / "stirnet" / "first_overfit" / "10_corrected_same_sample"
CHECKPOINT_PATH = RUN10_DIR / "checkpoint_step_025.pt"
RUN11_DIR = REPO_ROOT / "runs" / "stirnet" / "debugging" / "11_stage_b_existence_and_correction"
MASK_CACHE_DIR = RUN11_DIR / "query_mask_cache"
RUN11_DIR.mkdir(parents=True, exist_ok=True)
MASK_CACHE_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 11 requires CUDA.")
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(CHECKPOINT_PATH)

device = torch.device("cuda")
print("Repo:", REPO_ROOT)
print("Checkpoint:", CHECKPOINT_PATH)
print("GPU:", torch.cuda.get_device_name(0))


## 1. Rebuild the exact scene and load Notebook-10 step 25

In [ ]:
batch, sample = build_real_batch(DATA_DIR)
cfg = _reduced_config()
target = batch["targets"][0]

assert sample["current_count"] == 36
assert sample["target_count"] == 33
assert sample["temporal_tracklets"] == 52
assert sample["required_queries"] == 132
assert "source_ids" in target and "source_gt_overlap" in target

def prepare_device_batch(cpu_batch):
    out={}
    for k,v in cpu_batch.items():
        if k=="targets":
            out[k]=v
        elif k=="spatial_inputs":
            out[k]=v.to(device=device,dtype=AMP_DTYPE,non_blocking=True)
        elif k=="instance_labels":
            out[k]=v.to(device=device,dtype=torch.int32,non_blocking=True)
        else:
            out[k]=move_to_device(v,device)
    return out

def forward_debug(model,b):
    return model(
        b["spatial_inputs"], b["instance_labels"], b["spacing_um"], b["dref_um"],
        b["instance_features"], b["instance_ids"], b["instance_batch"], b["instance_centroids_um"],
        b["graph_x"], b["graph_edge_index"], b["graph_edge_attr"], b["tracklet_id"],
        b["temporal_ref_um"], b["temporal_status"], b["hypothesis_edge_index"],
        b["hypothesis_edge_attr"], b["temporal_batch"], b.get("spatial_padding_mask"),
        return_debug=True,
    )

device_batch = prepare_device_batch(batch)
model = StirNet(cfg).to(device)
load_checkpoint(CHECKPOINT_PATH, model, map_location=device, strict=True)
model.eval()

criterion = RefinementCriterion(cfg.losses,cfg.queries,cfg.training).to(device)
criterion.eval()

gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
t0=time.perf_counter()
with torch.no_grad(), torch.autocast(device_type="cuda",dtype=AMP_DTYPE):
    outputs=forward_debug(model,device_batch)
    losses=criterion(outputs,device_batch["targets"])
torch.cuda.synchronize()

print("Scene:",sample)
print("Spacing:",batch["spacing_um"][0].tolist(),"dref:",float(batch["dref_um"][0]))
print("Loss:",float(losses["loss"].detach().float().cpu()))
print("Forward+loss s:",time.perf_counter()-t0)
print("Peak CUDA GiB:",torch.cuda.max_memory_allocated()/1024**3)


## 2. Definitive structured matching

In [ ]:
probe=run_matching_probe(outputs,[target])
match=probe.matches[0]
matched_q=match.pred_indices.detach().cpu().long()
matched_t=match.target_indices.detach().cpu().long()

qtypes=outputs.query_types[0].detach().cpu().long()
source_ids_q=outputs.source_instance_ids[0].detach().cpu().long()
padding=outputs.query_padding_mask[0].detach().cpu().bool()
valid_q_mask=~padding
exist_prob=torch.sigmoid(outputs.exist_logits[0]).detach().float().cpu()

matched_map={int(q):int(t) for q,t in zip(matched_q.tolist(),matched_t.tolist())}
matched_set=set(matched_map)

query_rows=[]
for q in torch.nonzero(valid_q_mask,as_tuple=False).flatten().tolist():
    q=int(q)
    query_rows.append({
        "query":q,
        "query_type":QUERY_NAMES[int(qtypes[q])],
        "source_instance_id":int(source_ids_q[q]),
        "matched":q in matched_map,
        "target_index":matched_map.get(q,-1),
        "exist_prob":float(exist_prob[q]),
        "passes_0.30":bool(exist_prob[q]>0.30),
        "passes_0.50":bool(exist_prob[q]>0.50),
    })
query_df=pd.DataFrame(query_rows)

display(query_df.groupby("query_type").agg(
    total=("query","count"),
    matched=("matched","sum"),
    mean_exist=("exist_prob","mean"),
    above_030=("passes_0.30","sum"),
    above_050=("passes_0.50","sum"),
).reset_index())

print("Matched GT:",len(matched_q),"/",sample["target_count"])


## 3. Stage A vs Stage B

Stage A contains primary/split matches. Stage B contains temporal/discovery matches to GT cells left after Stage A.


In [ ]:
seeded_types={QUERY_PRIMARY,QUERY_SPLIT}
recovery_types={QUERY_TEMPORAL,QUERY_DISCOVERY}

stage_a_pairs=[(int(q),int(t)) for q,t in zip(matched_q.tolist(),matched_t.tolist()) if int(qtypes[q]) in seeded_types]
stage_b_pairs=[(int(q),int(t)) for q,t in zip(matched_q.tolist(),matched_t.tolist()) if int(qtypes[q]) in recovery_types]

stage_a_gt={t for _,t in stage_a_pairs}
all_valid_gt=valid_target_indices(target).detach().cpu().long().tolist()
remaining_gt=[int(t) for t in all_valid_gt if int(t) not in stage_a_gt]
recovery_q=[int(q) for q in torch.nonzero(valid_q_mask,as_tuple=False).flatten().tolist() if int(qtypes[q]) in recovery_types]

print("Stage A:",len(stage_a_pairs))
print("Stage B:",len(stage_b_pairs))
print("Remaining GT after Stage A:",len(remaining_gt))
print("Recovery candidates:",len(recovery_q))
assert set(remaining_gt)=={t for _,t in stage_b_pairs}


## 4. Reconstruct Stage-B cost components

This reproduces the current matcher terms for temporal/discovery queries against the GT cells remaining after Stage A:

`2 × existence + 5 × Dice + 2 × focal + 2 × center`

The table also records initial/final Euclidean center error and current native-support coverage.


In [ ]:
def pairwise_dice_cost(pred_logits,gt,support):
    p=pred_logits.float().sigmoid()
    gt=gt.float()
    inter=2*torch.einsum("qv,kv->qk",p,gt)
    pred_sum=torch.einsum("qv,kv->qk",p,support.float())
    denom=pred_sum+gt.sum(-1)[None]
    return 1-(inter+1e-6)/(denom+1e-6)

def pairwise_focal_cost(pred_logits,gt,support,alpha,gamma):
    logits=pred_logits.float()
    gt=gt.float()
    prob=logits.sigmoid()
    pos=-alpha*(1-prob).pow(gamma)*F.logsigmoid(logits)
    neg=-(1-alpha)*prob.pow(gamma)*F.logsigmoid(-logits)
    cols=[]
    for k in range(gt.shape[0]):
        value=pos*gt[k][None]+neg*(1-gt[k][None])
        local=support[k].float()[None]
        cols.append((value*local).sum(-1)/local.sum().clamp_min(1))
    return torch.stack(cols,dim=1)

recovery_idx=torch.as_tensor(recovery_q,device=device,dtype=torch.long)
remaining_idx=torch.as_tensor(remaining_gt,dtype=torch.long)

coarse_shape=tuple(int(v) for v in outputs.coarse_mask_logits.shape[-3:])
gt_masks=target_masks_at_shape(target,coarse_shape,device,target_indices=remaining_idx)

gt_centers_all=torch.as_tensor(target["centers_cellscale"]).float()
gt_centers=gt_centers_all[remaining_idx].to(device)

spacing=outputs.coarse_spacing_um
spacing_b=spacing[0] if spacing.ndim==2 else spacing
dref=outputs.dref_um[0]

gt_support=build_local_support_masks(
    gt_masks,gt_centers,spacing_b,dref,cfg.losses.mask_supervision_radius_dref
)

pred_coarse=outputs.coarse_mask_logits[0,recovery_idx].float().flatten(1)
gt_flat=gt_masks.float().flatten(1)
support_flat=gt_support.bool().flatten(1)

exist_c=-F.logsigmoid(outputs.exist_logits[0,recovery_idx].float())[:,None].expand(-1,len(remaining_gt))
dice_c=pairwise_dice_cost(pred_coarse,gt_flat,support_flat)
focal_c=pairwise_focal_cost(
    pred_coarse,gt_flat,support_flat,
    cfg.losses.mask_focal_alpha_pos,cfg.losses.mask_focal_gamma,
)
center_c=torch.cdist(
    outputs.centers_cellscale[0,recovery_idx].float(),
    gt_centers.float(),
    p=1,
)
total_c=2*exist_c+5*dice_c+2*focal_c+2*center_c

initial_refs=outputs.debug["query_initial_references_cellscale"][0].detach().float().cpu()
layer_refs=outputs.debug["query_layer_references_cellscale"][:,0].detach().float().cpu()
final_refs=outputs.centers_cellscale[0].detach().float().cpu()
dref_value=float(outputs.dref_um[0].detach().cpu())
gt_centers_cpu=gt_centers_all.cpu()

recovery_pos={q:i for i,q in enumerate(recovery_q)}
remaining_pos={t:j for j,t in enumerate(remaining_gt)}

stageb_rows=[]
for q,t in stage_b_pairs:
    i=recovery_pos[q]; j=remaining_pos[t]
    final_dist=float(torch.linalg.vector_norm(final_refs[q]-gt_centers_cpu[t])*dref_value)
    initial_dist=float(torch.linalg.vector_norm(initial_refs[q]-gt_centers_cpu[t])*dref_value)
    total_col=total_c[:,j].detach().cpu()
    center_l2_all=torch.linalg.vector_norm(
        final_refs[torch.as_tensor(recovery_q)]-gt_centers_cpu[t][None],dim=-1
    )*dref_value
    stageb_rows.append({
        "query":q,
        "query_type":QUERY_NAMES[int(qtypes[q])],
        "target_index":t,
        "exist_prob":float(exist_prob[q]),
        "initial_error_um":initial_dist,
        "layer1_error_um":float(torch.linalg.vector_norm(layer_refs[0,q]-gt_centers_cpu[t])*dref_value),
        "layer2_error_um":float(torch.linalg.vector_norm(layer_refs[1,q]-gt_centers_cpu[t])*dref_value),
        "final_error_um":final_dist,
        "final_error_dref":final_dist/dref_value,
        "exist_cost":float(exist_c[i,j].detach().cpu()),
        "dice_cost":float(dice_c[i,j].detach().cpu()),
        "focal_cost":float(focal_c[i,j].detach().cpu()),
        "center_L1_cost":float(center_c[i,j].detach().cpu()),
        "total_cost":float(total_c[i,j].detach().cpu()),
        "total_cost_rank_for_gt":int((total_col<total_col[i].cpu()).sum())+1,
        "center_distance_rank_for_gt":int((center_l2_all<center_l2_all[i]).sum())+1,
    })

stageb_df=pd.DataFrame(stageb_rows)
display(stageb_df.sort_values("final_error_um",ascending=False))


## 5. Exact native render-support reachability for Stage-B matches

In [ ]:
gt_label_map=torch.as_tensor(target["label_map"]).cpu().numpy().astype(np.int32,copy=False)
gt_ids=np.asarray(torch.as_tensor(target["ids"]).cpu(),dtype=np.int64)
spacing_native=np.asarray(outputs.spacing_um[0].detach().float().cpu(),dtype=np.float32)
shape=np.asarray(gt_label_map.shape,dtype=np.int64)
extent=(shape-1)*spacing_native
radius_um=float(cfg.queries.native_support_radius_dref)*dref_value

def gt_radial_coverage(target_index,ref_cellscale):
    gt_id=int(gt_ids[target_index])
    coords=np.argwhere(gt_label_map==gt_id).astype(np.float32)
    if len(coords)==0:
        return np.nan
    coords_um=coords*spacing_native[None]-0.5*extent[None]
    ref_um=np.asarray(ref_cellscale,dtype=np.float32)*dref_value
    dist2=((coords_um-ref_um[None])**2).sum(axis=1)
    return float((dist2<=radius_um**2).mean())

coverage=[]
for _,r in stageb_df.iterrows():
    q=int(r["query"]); t=int(r["target_index"])
    coverage.append(gt_radial_coverage(t,final_refs[q].numpy()))
stageb_df["native_support_gt_coverage"]=coverage
stageb_df["support_compatible_95pct"]=stageb_df["native_support_gt_coverage"]>=0.95

display(stageb_df[[
    "query","query_type","target_index","initial_error_um","final_error_um",
    "final_error_dref","native_support_gt_coverage","total_cost_rank_for_gt",
    "center_distance_rank_for_gt"
]].sort_values("native_support_gt_coverage"))

stageb_df.to_csv(RUN11_DIR/"stage_b_matches.csv",index=False)


## 6. Offline Stage-B hard-gate simulation

No model source is changed here.

For a proposed maximum **final-center distance** gate, we solve a maximum-cardinality, minimum-cost assignment. A GT is allowed to remain unmatched rather than being forced onto an ineligible recovery query.


In [ ]:
final_distance_matrix=torch.empty((len(recovery_q),len(remaining_gt)),dtype=torch.float32)
initial_distance_matrix=torch.empty_like(final_distance_matrix)

recovery_tensor_cpu=torch.as_tensor(recovery_q,dtype=torch.long)
for j,t in enumerate(remaining_gt):
    final_distance_matrix[:,j]=torch.linalg.vector_norm(
        final_refs[recovery_tensor_cpu]-gt_centers_cpu[t][None],dim=-1
    )*dref_value
    initial_distance_matrix[:,j]=torch.linalg.vector_norm(
        initial_refs[recovery_tensor_cpu]-gt_centers_cpu[t][None],dim=-1
    )*dref_value

def gated_assignment(cost_cpu,dist_cpu,max_distance_um):
    cost=np.asarray(cost_cpu,dtype=np.float64)
    dist=np.asarray(dist_cpu,dtype=np.float64)
    Q,K=cost.shape
    eligible=dist<=max_distance_um

    finite_real=cost[eligible]
    match_penalty=(finite_real.max()+1.0) if finite_real.size else 1.0
    invalid=match_penalty*(Q+K+10)

    aug=np.full((Q+K,K),invalid,dtype=np.float64)
    aug[:Q]=np.where(eligible,cost,invalid)
    for k in range(K):
        aug[Q+k,k]=match_penalty

    rows,cols=linear_sum_assignment(aug)
    real=[(int(r),int(c)) for r,c in zip(rows,cols) if r<Q and eligible[r,c]]
    return real

gate_rows=[]
for basis_name,dist_matrix in [
    ("final",final_distance_matrix),
    ("initial",initial_distance_matrix),
]:
    for gate_dref in [0.5,1.0,1.5,2.0,2.5,3.0]:
        pairs=gated_assignment(
            total_c.detach().cpu().numpy(),
            dist_matrix.numpy(),
            gate_dref*dref_value,
        )
        dists=[float(dist_matrix[i,j]) for i,j in pairs]
        types=[QUERY_NAMES[int(qtypes[recovery_q[i]])] for i,j in pairs]
        gate_rows.append({
            "distance_basis":basis_name,
            "gate_dref":gate_dref,
            "matched_gt":len(pairs),
            "unmatched_gt":len(remaining_gt)-len(pairs),
            "temporal_matches":types.count("temporal"),
            "discovery_matches":types.count("discovery"),
            "mean_distance_um":float(np.mean(dists)) if dists else np.nan,
            "max_distance_um":float(np.max(dists)) if dists else np.nan,
        })

gate_df=pd.DataFrame(gate_rows)
display(gate_df)
gate_df.to_csv(RUN11_DIR/"stage_b_gate_simulation.csv",index=False)


## 7. Memory-safe query-mask cache

We render each required query only once and save the postprocessed connected component sparsely (`flat voxel indices + probabilities`).

The cache contains:
- every Hungarian-matched query, regardless of existence;
- every valid query with existence > 0.10, for threshold sweeps.

This keeps repeated diagnostics cheap and avoids storing `[Q,Z,Y,X]`.


In [ ]:
def component_near_center(mask,center_vox,min_voxels):
    cc,count=ndi.label(mask)
    if count==0:
        return np.zeros_like(mask,dtype=bool)
    center=np.rint(center_vox).astype(int)
    if np.all(center>=0) and np.all(center<np.asarray(mask.shape)):
        label=int(cc[tuple(center)])
        if label>0 and np.count_nonzero(cc==label)>=min_voxels:
            return cc==label
    objects=ndi.find_objects(cc)
    best_label=None; best_dist=np.inf
    for label,slices in enumerate(objects,start=1):
        if slices is None: continue
        comp=cc[slices]==label
        if int(comp.sum())<min_voxels: continue
        coords=np.argwhere(comp)+np.asarray([s.start for s in slices])[None]
        d=float(np.linalg.norm(coords-center[None],axis=1).min())
        if d<best_dist:
            best_dist=d; best_label=label
    return cc==best_label if best_label is not None else np.zeros_like(mask,dtype=bool)

shape_tuple=tuple(int(v) for v in outputs.instance_labels.shape[-3:])
extent_um=(np.asarray(shape_tuple,dtype=np.float32)-1)*spacing_native
candidate_queries=sorted(
    set(matched_q.tolist()) |
    set(torch.nonzero(valid_q_mask & (exist_prob>0.10),as_tuple=False).flatten().tolist())
)
print("Queries to cache:",len(candidate_queries))

@torch.no_grad()
def render_and_cache_query(q):
    path=MASK_CACHE_DIR/f"q_{q:03d}.npz"
    if path.exists():
        return
    idx=torch.tensor([q],device=device,dtype=torch.long)
    rendered=model.render_masks(outputs,[idx])[0][0]
    prob=torch.sigmoid(rendered.float()).cpu().numpy().astype(np.float32,copy=False)
    binary=prob>cfg.inference.mask_threshold

    center_rel_um=outputs.centers_cellscale[0,q].detach().float().cpu().numpy()*dref_value
    center_vox=(center_rel_um+0.5*extent_um)/spacing_native
    binary=component_near_center(binary,center_vox,cfg.inference.min_mask_voxels)

    flat_idx=np.flatnonzero(binary.reshape(-1)).astype(np.int32)
    flat_prob=prob.reshape(-1)[flat_idx].astype(np.float16)
    np.savez_compressed(path,indices=flat_idx,probabilities=flat_prob)

    del rendered,prob,binary,flat_idx,flat_prob
    gc.collect(); torch.cuda.empty_cache()

for n,q in enumerate(candidate_queries,1):
    render_and_cache_query(int(q))
    if n%10==0 or n==len(candidate_queries):
        print(f"cached {n}/{len(candidate_queries)}")


## 8. Sparse assembly and evaluation helpers

In [ ]:
gt_flat=gt_label_map.reshape(-1)
gt_fg=gt_flat>0
voxel_count=gt_flat.size

def load_sparse(q):
    data=np.load(MASK_CACHE_DIR/f"q_{int(q):03d}.npz")
    return data["indices"].astype(np.int64,copy=False),data["probabilities"].astype(np.float32,copy=False)

def assemble_query_set(query_ids,use_existence_score=True):
    label_flat=np.zeros(voxel_count,dtype=np.int32)
    best=np.full(voxel_count,-np.inf,dtype=np.float32)
    used=[]
    for label_id,q in enumerate(query_ids,start=1):
        path=MASK_CACHE_DIR/f"q_{int(q):03d}.npz"
        if not path.exists(): continue
        idx,prob=load_sparse(q)
        if len(idx)<cfg.inference.min_mask_voxels: continue
        score=prob*float(exist_prob[q]) if use_existence_score else prob
        keep=score>best[idx]
        upd=idx[keep]
        if len(upd):
            label_flat[upd]=label_id
            best[upd]=score[keep]
            used.append(int(q))
    return label_flat.reshape(shape_tuple),used

def instance_eval(pred,gt):
    pred_ids,pred_counts=np.unique(pred[pred>0],return_counts=True)
    gt_ids_unique,gt_counts=np.unique(gt[gt>0],return_counts=True)
    if len(pred_ids)==0 or len(gt_ids_unique)==0:
        return {
            "pred_count":len(pred_ids),"gt_count":len(gt_ids_unique),
            "mean_matched_dice":0.0,"median_matched_dice":0.0,
            "positive_pairs":0,"false_positive_instances":len(pred_ids),
            "foreground_dice":0.0,
        }
    pr={int(x):i for i,x in enumerate(pred_ids)}
    gr={int(x):i for i,x in enumerate(gt_ids_unique)}
    inter=np.zeros((len(pred_ids),len(gt_ids_unique)),dtype=np.int64)
    pos=(pred>0)&(gt>0)
    if pos.any():
        pairs,counts=np.unique(np.stack([pred[pos],gt[pos]],axis=1),axis=0,return_counts=True)
        for (p,g),c in zip(pairs.tolist(),counts.tolist()):
            inter[pr[int(p)],gr[int(g)]]=int(c)
    dice=2*inter/np.maximum(pred_counts[:,None]+gt_counts[None,:],1)
    rr,cc=linear_sum_assignment(1-dice)
    vals=dice[rr,cc]
    pred_has_overlap=(inter>0).any(axis=1)
    pred_fg=(pred>0)
    fg_dice=2*np.count_nonzero(pred_fg&(gt>0))/max(np.count_nonzero(pred_fg)+np.count_nonzero(gt>0),1)
    return {
        "pred_count":int(len(pred_ids)),
        "gt_count":int(len(gt_ids_unique)),
        "mean_matched_dice":float(vals.mean()) if len(vals) else 0.0,
        "median_matched_dice":float(np.median(vals)) if len(vals) else 0.0,
        "positive_pairs":int(np.count_nonzero(vals>0)),
        "false_positive_instances":int((~pred_has_overlap).sum()),
        "foreground_dice":float(fg_dice),
    }


## 9. Existence-threshold sweep + oracle matched output

The **oracle matched output** uses exactly the 33 Hungarian-matched queries and ignores existence in both filtering and voxel competition. It is not a deployable metric; it measures what the mask branch can currently provide if query selection were perfect.


In [ ]:
oracle_labels,oracle_used=assemble_query_set(matched_q.tolist(),use_existence_score=False)
oracle_metrics=instance_eval(oracle_labels,gt_label_map)
print("Oracle matched-query output:")
print(oracle_metrics)

thresholds=[0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.45,0.48,0.49,0.50,0.52,0.55]
sweep_rows=[]
best_labels=None; best_metric=-1; best_threshold=None

for thr in thresholds:
    selected=[
        int(q) for q in torch.nonzero(valid_q_mask,as_tuple=False).flatten().tolist()
        if float(exist_prob[q])>thr and (MASK_CACHE_DIR/f"q_{int(q):03d}.npz").exists()
    ]
    labels,used=assemble_query_set(selected,use_existence_score=True)
    metrics=instance_eval(labels,gt_label_map)
    type_counts={name:0 for name in QUERY_NAMES.values()}
    for q in used: type_counts[QUERY_NAMES[int(qtypes[q])]]+=1
    row={"threshold":thr,"selected_queries":len(selected),"used_nonempty":len(used),**metrics,**{f"used_{k}":v for k,v in type_counts.items()}}
    sweep_rows.append(row)
    if metrics["mean_matched_dice"]>best_metric:
        best_metric=metrics["mean_matched_dice"]; best_threshold=thr; best_labels=labels.copy()

sweep_df=pd.DataFrame(sweep_rows)
display(sweep_df)
sweep_df.to_csv(RUN11_DIR/"existence_threshold_sweep.csv",index=False)

print("Best diagnostic threshold by mean matched Dice:",best_threshold)


In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(sweep_df["threshold"],sweep_df["pred_count"],marker="o",label="predicted instances")
ax.plot(sweep_df["threshold"],sweep_df["positive_pairs"],marker="o",label="positive GT↔pred pairs")
ax.axvline(0.50,linestyle="--",label="production 0.50")
ax.set_xlabel("Existence threshold"); ax.set_ylabel("Count")
ax.set_title("Existence threshold sweep")
ax.legend(); ax.grid(alpha=0.2); plt.show()

fig,ax=plt.subplots(figsize=(9,5))
ax.plot(sweep_df["threshold"],sweep_df["mean_matched_dice"],marker="o",label="mean matched Dice")
ax.plot(sweep_df["threshold"],sweep_df["foreground_dice"],marker="o",label="foreground Dice")
ax.axvline(0.50,linestyle="--",label="production 0.50")
ax.set_xlabel("Existence threshold"); ax.set_ylabel("Score")
ax.set_title("Segmentation quality vs existence threshold")
ax.legend(); ax.grid(alpha=0.2); plt.show()


## 10. Is STIR-Net truly correcting the current CC input?

For every matched primary/split query:

`improvement = Dice(STIR-Net mask, matched GT) - Dice(current source, matched GT)`

Positive values mean the query mask improves its source component for that GT.


In [ ]:
current_labels=batch["instance_labels"][0].cpu().numpy().astype(np.int32,copy=False)
current_flat=current_labels.reshape(-1)

source_counts=np.bincount(current_flat)
gt_count_by_id={int(g):int(np.count_nonzero(gt_flat==g)) for g in gt_ids}

positive=(current_flat>0)&(gt_flat>0)
pair_counts={}
if positive.any():
    pairs,counts=np.unique(np.stack([current_flat[positive],gt_flat[positive]],axis=1),axis=0,return_counts=True)
    pair_counts={(int(s),int(g)):int(c) for (s,g),c in zip(pairs.tolist(),counts.tolist())}

def source_gt_dice(source_id,target_index):
    gid=int(gt_ids[target_index])
    inter=pair_counts.get((int(source_id),gid),0)
    sc=int(source_counts[source_id]) if 0<=source_id<len(source_counts) else 0
    gc=gt_count_by_id[gid]
    return 2*inter/max(sc+gc,1)

def sparse_query_gt_dice(q,target_index):
    idx,_=load_sparse(q)
    gid=int(gt_ids[target_index])
    inter=int(np.count_nonzero(gt_flat[idx]==gid))
    return 2*inter/max(len(idx)+gt_count_by_id[gid],1)

source_ids_meta=torch.as_tensor(target["source_ids"]).cpu().long()
overlap_meta=torch.as_tensor(target["source_gt_overlap"]).cpu()
source_row={int(s):i for i,s in enumerate(source_ids_meta.tolist())}

corr_rows=[]
for q,t in zip(matched_q.tolist(),matched_t.tolist()):
    q=int(q); t=int(t); typ=int(qtypes[q])
    if typ not in seeded_types: continue
    sid=int(source_ids_q[q])
    n_overlap=int((overlap_meta[source_row[sid]]>0).sum()) if sid in source_row else 0
    before=source_gt_dice(sid,t)
    after=sparse_query_gt_dice(q,t)
    corr_rows.append({
        "query":q,"query_type":QUERY_NAMES[typ],"source_instance_id":sid,
        "source_overlap_gt_count":n_overlap,"target_index":t,
        "source_to_gt_dice":before,"prediction_to_gt_dice":after,
        "improvement":after-before,"exist_prob":float(exist_prob[q]),
    })

correction_df=pd.DataFrame(corr_rows)
display(correction_df.sort_values("improvement"))
print("\nSummary by source difficulty:")
display(correction_df.groupby(["query_type","source_overlap_gt_count"]).agg(
    count=("query","count"),
    mean_source_dice=("source_to_gt_dice","mean"),
    mean_prediction_dice=("prediction_to_gt_dice","mean"),
    mean_improvement=("improvement","mean"),
    improved=("improvement",lambda x:int((x>0).sum())),
).reset_index())

correction_df.to_csv(RUN11_DIR/"seeded_true_correction.csv",index=False)


## 11. Merged-source primary/split analysis

For each source overlapping two or more GT cells, compare:
- source→each GT Dice;
- primary/split prediction→each compatible GT Dice;
- primary-vs-split mask overlap.

A useful split should produce two distinct masks, not two copies of the merged source.


In [ ]:
def sparse_pair_dice(q1,q2):
    i1,_=load_sparse(q1); i2,_=load_sparse(q2)
    inter=len(np.intersect1d(i1,i2,assume_unique=True))
    return 2*inter/max(len(i1)+len(i2),1)

merge_rows=[]
merge_summary=[]
for sid,row_idx in source_row.items():
    compatible=torch.nonzero(overlap_meta[row_idx]>0,as_tuple=False).flatten().tolist()
    if len(compatible)<2: continue

    source_queries=[
        int(q) for q in torch.nonzero(valid_q_mask,as_tuple=False).flatten().tolist()
        if int(source_ids_q[q])==sid and int(qtypes[q]) in seeded_types
    ]
    primary=[q for q in source_queries if int(qtypes[q])==QUERY_PRIMARY]
    split=[q for q in source_queries if int(qtypes[q])==QUERY_SPLIT]

    for t in compatible:
        base=source_gt_dice(sid,int(t))
        for q in source_queries:
            merge_rows.append({
                "source_instance_id":sid,
                "compatible_gt_index":int(t),
                "query":q,
                "query_type":QUERY_NAMES[int(qtypes[q])],
                "query_is_matched_to_this_gt":matched_map.get(q,-1)==int(t),
                "source_to_gt_dice":base,
                "query_to_gt_dice":sparse_query_gt_dice(q,int(t)),
                "exist_prob":float(exist_prob[q]),
            })

    overlap_ps=np.nan
    if primary and split:
        overlap_ps=sparse_pair_dice(primary[0],split[0])
    merge_summary.append({
        "source_instance_id":sid,
        "compatible_gt_count":len(compatible),
        "primary_query":primary[0] if primary else -1,
        "split_query":split[0] if split else -1,
        "primary_split_mask_dice":overlap_ps,
        "primary_match":matched_map.get(primary[0],-1) if primary else -1,
        "split_match":matched_map.get(split[0],-1) if split else -1,
    })

merge_df=pd.DataFrame(merge_rows)
merge_summary_df=pd.DataFrame(merge_summary)
print("Merged sources:")
display(merge_summary_df)
print("Per-query / per-compatible-GT:")
display(merge_df.sort_values(["source_instance_id","compatible_gt_index","query_type"]))

merge_summary_df.to_csv(RUN11_DIR/"merged_source_summary.csv",index=False)
merge_df.to_csv(RUN11_DIR/"merged_source_query_gt_dice.csv",index=False)


## 12. Existence-loss vs count-loss gradients on final logits

We differentiate only with respect to the final existence logits. This cleanly answers whether, at this checkpoint, weighted count consistency pushes matched queries in the opposite direction from weighted existence focal supervision.


In [ ]:
logits=outputs.exist_logits.detach().float().clone().requires_grad_(True)
pad=outputs.query_padding_mask.detach()
valid=~pad
exist_target=torch.zeros_like(logits)
exist_target[0,match.pred_indices]=1.0

exist_loss=binary_focal_loss_with_logits(
    logits[valid],exist_target[valid],
    alpha=cfg.losses.exist_focal_alpha_pos,
    gamma=cfg.losses.exist_focal_gamma,
)
prob=torch.sigmoid(logits).masked_fill(pad,0)
predicted_count=prob.sum(-1)
target_count=torch.tensor([sample["target_count"]],device=device,dtype=logits.dtype)
count_loss=(F.smooth_l1_loss(predicted_count,target_count,reduction="none")/target_count.clamp_min(1)).mean()

weighted_exist=cfg.losses.exist*exist_loss
weighted_count=cfg.losses.count*count_loss

g_exist=torch.autograd.grad(weighted_exist,logits,retain_graph=True)[0].detach().cpu()[0]
g_count=torch.autograd.grad(weighted_count,logits)[0].detach().cpu()[0]

valid_cpu=valid[0].cpu()
matched_bool=torch.tensor([q in matched_set for q in range(len(qtypes))],dtype=torch.bool)

def cosine(a,b,mask):
    aa=a[mask].float(); bb=b[mask].float()
    return float(torch.dot(aa,bb)/(torch.linalg.vector_norm(aa)*torch.linalg.vector_norm(bb)+1e-12))

grad_rows=[]
for q in torch.nonzero(valid_q_mask,as_tuple=False).flatten().tolist():
    q=int(q)
    grad_rows.append({
        "query":q,"query_type":QUERY_NAMES[int(qtypes[q])],"matched":bool(matched_bool[q]),
        "exist_prob":float(exist_prob[q]),
        "weighted_exist_grad":float(g_exist[q]),
        "weighted_count_grad":float(g_count[q]),
        "weighted_combined_grad":float(g_exist[q]+g_count[q]),
        "exist_descent_delta":float(-g_exist[q]),
        "count_descent_delta":float(-g_count[q]),
    })
grad_df=pd.DataFrame(grad_rows)

print("Predicted expected count:",float(predicted_count.detach().cpu()[0]))
print("Target count:",sample["target_count"])
print("Weighted exist loss:",float(weighted_exist.detach().cpu()))
print("Weighted count loss:",float(weighted_count.detach().cpu()))
print("Gradient cosine, all valid:",cosine(g_exist,g_count,valid_q_mask))
print("Gradient cosine, matched:",cosine(g_exist,g_count,valid_q_mask & matched_bool))
print("Gradient cosine, unmatched:",cosine(g_exist,g_count,valid_q_mask & ~matched_bool))

display(grad_df.groupby(["query_type","matched"]).agg(
    count=("query","count"),
    mean_prob=("exist_prob","mean"),
    mean_exist_grad=("weighted_exist_grad","mean"),
    mean_count_grad=("weighted_count_grad","mean"),
    mean_combined_grad=("weighted_combined_grad","mean"),
).reset_index())

grad_df.to_csv(RUN11_DIR/"existence_count_logit_gradients.csv",index=False)


## 13. Linear probe on final query embeddings

This asks whether the final query representation already contains linearly accessible information about matched vs unmatched status.

We report:
- raw existence-logit ROC AUC;
- 5-fold cross-validated linear-probe ROC AUC.

High probe AUC but weak raw existence AUC suggests the representation contains the information and the current existence head/optimization is the problem. Weak probe AUC suggests the query representation itself does not clearly encode cell/no-object status.


In [ ]:
features=outputs.query_embeddings[0].detach().float().cpu()[valid_q_mask].numpy()
labels=np.asarray([1 if int(q) in matched_set else 0 for q in torch.nonzero(valid_q_mask,as_tuple=False).flatten().tolist()],dtype=np.int64)
raw_scores=outputs.exist_logits[0].detach().float().cpu()[valid_q_mask].numpy()

def roc_auc_rank(y,s):
    y=np.asarray(y); s=np.asarray(s)
    pos=y==1; neg=y==0
    if pos.sum()==0 or neg.sum()==0: return np.nan
    ranks=rankdata(s,method="average")
    return float((ranks[pos].sum()-pos.sum()*(pos.sum()+1)/2)/(pos.sum()*neg.sum()))

raw_auc=roc_auc_rank(labels,raw_scores)
print("Raw existence-logit AUC:",raw_auc)

rng=np.random.default_rng(SEED)
pos_idx=np.flatnonzero(labels==1); neg_idx=np.flatnonzero(labels==0)
rng.shuffle(pos_idx); rng.shuffle(neg_idx)
pos_folds=np.array_split(pos_idx,5); neg_folds=np.array_split(neg_idx,5)

probe_rows=[]
for fold in range(5):
    test_idx=np.concatenate([pos_folds[fold],neg_folds[fold]])
    train_idx=np.setdiff1d(np.arange(len(labels)),test_idx)

    mu=features[train_idx].mean(0,keepdims=True)
    sd=features[train_idx].std(0,keepdims=True)+1e-6
    xtr=torch.tensor((features[train_idx]-mu)/sd,dtype=torch.float32)
    ytr=torch.tensor(labels[train_idx],dtype=torch.float32)
    xte=torch.tensor((features[test_idx]-mu)/sd,dtype=torch.float32)

    torch.manual_seed(SEED+fold)
    probe_model=nn.Linear(features.shape[1],1)
    opt=torch.optim.AdamW(probe_model.parameters(),lr=0.03,weight_decay=1e-2)
    pos_weight=torch.tensor([(ytr.numel()-ytr.sum())/ytr.sum().clamp_min(1)])

    for _ in range(300):
        opt.zero_grad(set_to_none=True)
        z=probe_model(xtr).squeeze(-1)
        loss=F.binary_cross_entropy_with_logits(z,ytr,pos_weight=pos_weight)
        loss.backward(); opt.step()

    with torch.no_grad():
        score=probe_model(xte).squeeze(-1).numpy()
    auc=roc_auc_rank(labels[test_idx],score)
    probe_rows.append({"fold":fold,"test_n":len(test_idx),"auc":auc})

probe_df=pd.DataFrame(probe_rows)
display(probe_df)
print("5-fold linear-probe mean AUC:",probe_df["auc"].mean())
probe_df.to_csv(RUN11_DIR/"linear_probe_cv.csv",index=False)


## 14. Automatic diagnostic summary

These are evidence flags, not final product criteria.


In [ ]:
normal_050=sweep_df.iloc[(sweep_df["threshold"]-0.50).abs().argmin()]
best=sweep_df.iloc[sweep_df["mean_matched_dice"].argmax()]

stageb_far_fraction=float((stageb_df["final_error_dref"]>cfg.queries.native_support_radius_dref).mean()) if len(stageb_df) else 0.0
stageb_low_coverage_fraction=float((stageb_df["native_support_gt_coverage"]<0.95).mean()) if len(stageb_df) else 0.0

matched_grad=grad_df[grad_df["matched"]]
count_opposes_exist=float(
    (np.sign(matched_grad["weighted_exist_grad"])!=np.sign(matched_grad["weighted_count_grad"])).mean()
)

summary={
    "oracle_matched":oracle_metrics,
    "normal_threshold_0_50":normal_050.to_dict(),
    "best_diagnostic_threshold":float(best_threshold),
    "best_threshold_metrics":best.to_dict(),
    "stage_b_match_count":len(stageb_df),
    "stage_b_fraction_beyond_native_radius":stageb_far_fraction,
    "stage_b_fraction_below_95pct_support_coverage":stageb_low_coverage_fraction,
    "expected_count":float(predicted_count.detach().cpu()[0]),
    "target_count":sample["target_count"],
    "matched_fraction_count_gradient_opposes_exist_gradient":count_opposes_exist,
    "raw_existence_auc":raw_auc,
    "linear_probe_mean_auc":float(probe_df["auc"].mean()),
    "seeded_mean_improvement":float(correction_df["improvement"].mean()) if len(correction_df) else np.nan,
    "seeded_improved_count":int((correction_df["improvement"]>0).sum()) if len(correction_df) else 0,
}

print(json.dumps(summary,indent=2,default=float))

with (RUN11_DIR/"summary.json").open("w",encoding="utf-8") as f:
    json.dump(summary,f,indent=2,default=float)


## 15. Optional Napari comparison

This opens four layers:

- Current CC input
- GT
- Oracle matched masks (existence ignored)
- Best diagnostic-threshold output

The oracle layer is especially useful for judging the current mask branch independently of existence selection.


In [ ]:
OPEN_NAPARI=True

if OPEN_NAPARI:
    import napari
    raw=batch["spatial_inputs"][0,0].float().cpu().numpy()
    spacing_zyx=tuple(float(v) for v in batch["spacing_um"][0])
    viewer=napari.Viewer(ndisplay=3)
    viewer.add_image(raw,name="Raw",scale=spacing_zyx)
    viewer.add_labels(current_labels,name="Current CC input",scale=spacing_zyx)
    viewer.add_labels(gt_label_map,name="GT",scale=spacing_zyx)
    viewer.add_labels(oracle_labels,name="Oracle matched - ignore existence",scale=spacing_zyx)
    viewer.add_labels(best_labels,name=f"Best threshold {best_threshold:.2f}",scale=spacing_zyx,visible=False)
    napari.run()
else:
    print("Napari skipped. Set OPEN_NAPARI=True to inspect.")


# How to interpret Notebook 11

## A. Existence is the main final-inference bottleneck if

- oracle matched output is much better than the 0.50 output;
- lower thresholds recover many valid instances;
- raw existence AUC is weak but the linear-probe AUC is strong.

Then the next patch should focus on existence-head optimization/calibration and the count interaction.

## B. Stage-B matching is forcing implausible assignments if

- many temporal/discovery matches are farther than the native support radius;
- many have very low GT support coverage;
- a reasonable offline distance gate leaves some GT unmatched instead of making absurd assignments.

Then Stage B needs explicit eligibility/no-match semantics before curriculum training.

## C. Visible primary masks are mostly source preservation if

- `source_to_gt_dice` is already high;
- `prediction_to_gt_dice` is similar;
- mean improvement is near zero.

That is not bad for clean cells, but it means the Napari layer should not be interpreted as proof of merge correction.

## D. Merge recovery is working if

- primary and split masks each favor different compatible GT cells;
- both improve substantially over the merged source→GT Dice;
- primary-vs-split mask Dice is low enough to indicate distinct daughter masks.

## E. Count is materially fighting existence if

- expected count remains above target;
- weighted count gradients oppose weighted existence gradients for a large fraction of matched queries;
- their magnitudes are non-negligible compared with existence gradients.

After this notebook, make the smallest source patch justified by the evidence, then implement staged/curriculum training in the following overfit notebook.
